# Train a fidelity kernel

Build a trainable similarity function and fit its Gram matrix to a label-derived target. You will evaluate it on both training pairs and new inputs.

Run the cells in order with QASP 0.1.2. See the [tutorial setup guide](../README.md) for installation and optional PDF rendering.

In [ ]:
%matplotlib inline
from pathlib import Path
from importlib.metadata import version
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

print("QASP", version("qasp"))
np.random.seed(17)
torch.manual_seed(17)

## Configure the experiment

The fidelity kernel encodes each input into a state. Each entry is the squared overlap between two encoded states. Using two reuploading layers allows the trainable circuit to change pairwise similarities.

In [ ]:
from qasp import Normal, Uniform
from qasp import create_kernel, KernelTargetAlignment, target_alignment, draw_kernel

torch.manual_seed(2026)
X_train = np.array([[-0.8, -0.5], [-0.2, -0.7],
                    [0.4, 0.8], [0.9, 0.3]], dtype=float)
y_train = np.array([0, 0, 1, 1])
X_test = np.array([[-0.6, -0.3], [0.6, 0.5]], dtype=float)
config = {
    "kernel_type": "fidelity",
    "feature_map": "AngleEmbedding",
    "feature_map_config": {"features": 2, "rotations": "Y", "input_scaling": True},
    "ansatz": "AngleAnsatz",
    "ansatz_config": {
        "n_layers": 2, "rotations": "YZ", "reuploading": True,
        "include_imprimitive": True, "imprimitive": "CZ",
        "imprimitive_pattern": "chain",
    },
    "init": Normal(0, 0.1), "input_scaling_init": Uniform(0.8, 1.2),
    "device": "default.qubit",
    "diff_method": "backprop", "shots": None,
}

## Construct and inspect the initial model

Run this cell again after editing the configuration to sample fresh parameters.

In [ ]:
kernel = create_kernel(config, n_qubits=2)
initial_gram = kernel.gram(X_train)
print("Initial alignment:", target_alignment(initial_gram, y_train))
print("Trainable tensors:", len(kernel.parameters()))
print("Initial angles:", kernel.qlayer.params.detach())
print("Initial encoding weights:", kernel.qlayer.input_scaling_params.detach())

## Train

KernelTargetAlignment adjusts the circuit to align its centered training Gram matrix with the label target. Keep the test inputs out of this optimization.

In [ ]:
alignment_trainer = KernelTargetAlignment(kernel, centered=True)
history = alignment_trainer.fit(
    X_train, y_train, steps=10, lr=0.03, seed=2026, verbose=False,
)
print("Alignment history:", history["alignment"])

## Read the results

Rows and columns of the square matrix index training observations. The rectangular test matrix instead has one row per test input and one column per training input.

In [ ]:
K_train = kernel.gram(X_train)
K_test = kernel.gram(X_test, X_train)
final_alignment = target_alignment(K_train, y_train)
print("Gram shapes:", K_train.shape, K_test.shape)
print("Final alignment:", final_alignment)
K_torch = kernel.gram(X_train, return_tensor=True)
print("Parameter-gradient graph:", K_torch.requires_grad)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(K_train, vmin=0, vmax=1, cmap="viridis")
fig.colorbar(im, ax=ax, label="Squared overlap")
ax.set(xlabel="Training input", ylabel="Training input")
plt.show()
np.testing.assert_allclose(K_train, K_train.T, atol=1e-6)
np.testing.assert_allclose(np.diag(K_train), 1, atol=1e-6)

## Inspect the circuit

Use the trained weights and the same representative input for both renderers. Matplotlib is the default for the module drawing helpers.

In [ ]:
fig, ax = draw_kernel(kernel=kernel, x=X_train[0], y=X_train[1])
display(fig)
plt.close(fig)

### Preview the paper figure

Request standalone TikZ for a vector PDF. Symbolic labels describe the circuit architecture. Set `symbolic_parameters=False` to inspect the gate angles at this input.

The helper uses temporary build files. To keep the displayed source, explicitly run `Path("circuit.tex").write_text(tex, encoding="utf-8")`, or call `builder.save_tikz(...)` when working with a builder.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import shutil
import subprocess
from IPython.display import Image, display


def show_tikz(source, name="Quantum circuit"):
    """Preview standalone TikZ inline. The optional name is the image description."""
    gs = shutil.which("gs") or shutil.which("gswin64c")
    if not shutil.which("pdflatex") or not gs:
        print("TikZ preview requires pdfLaTeX with Quantikz and Ghostscript.")
        return

    with TemporaryDirectory() as tmp:
        (Path(tmp) / "circuit.tex").write_text(source, encoding="utf-8")
        latex = subprocess.run(
            ["pdflatex", "-no-shell-escape", "-interaction=nonstopmode",
             "-halt-on-error", "circuit.tex"], cwd=tmp, capture_output=True, text=True,
        )
        if latex.returncode:
            raise RuntimeError(latex.stdout[-2000:])
        png = subprocess.check_output(
            [gs, "-q", "-dSAFER", "-sDEVICE=png16m", "-dTextAlphaBits=4", "-dGraphicsAlphaBits=4", "-r300", "-o", "-", "circuit.pdf"],
            cwd=tmp,
        )
    # Keep the original display width. Specifying only width preserves the aspect ratio.
    width_px = int.from_bytes(png[16:20], "big")  # PNG IHDR width
    display(Image(data=png, width=max(1, round(width_px * 120 / 300)), alt=name))

In [ ]:
tex = draw_kernel(
    kernel=kernel, x=X_train[0], y=X_train[1], output="tikz",
    tikz_options={"standalone": True, "symbolic_parameters": True, "row_sep": "0.7cm"},
)
show_tikz(tex, "kernel")

## Try it

Change the input-scale initializer to Ones(), rebuild the kernel, and compare the off-diagonal entries before and after alignment.